# 01 - Explore the PIT dataset

Reads the DuckDB views built by `pitlab data synthetic` or `pitlab data download`.
Light exploration only: the experiment itself is `pitlab run`. **Never commit outputs of this notebook when it runs on Sharadar data** (license).

In [ ]:
import duckdb, polars as pl
from pitlab.config import load_settings
s = load_settings()
con = duckdb.connect(str(s.duckdb_path), read_only=True)
con.sql('SHOW TABLES').pl()

## Rows per fundamentals dimension

In [ ]:
con.sql('SELECT dimension, count(*) n, count(DISTINCT ticker) tickers FROM fundamentals GROUP BY 1 ORDER BY 1').pl()

## Filing lag: days from period end (calendardate) to filing (datekey), ARQ
This is exactly the information arm A uses too early.

In [ ]:
con.sql("""SELECT quantile_cont(datekey - calendardate, [0.05,0.25,0.5,0.75,0.95]) AS lag_days_quantiles
FROM fundamentals WHERE dimension='ARQ'""").pl()

## How often do restated (MRQ) values differ from first-reported (ARQ)?

In [ ]:
con.sql("""
SELECT count(*) n_pairs,
       avg(CASE WHEN abs(a.netinc - m.netinc) > 1e-6*greatest(abs(a.netinc),1) THEN 1 ELSE 0 END) share_netinc_restated,
       avg(CASE WHEN abs(a.equity - m.equity) > 1e-6*greatest(abs(a.equity),1) THEN 1 ELSE 0 END) share_equity_restated
FROM fundamentals a JOIN fundamentals m
  ON a.ticker=m.ticker AND a.calendardate=m.calendardate
WHERE a.dimension='ARQ' AND m.dimension='MRQ'""").pl()

## Universe size over time (stocks with a price at each month end)

In [ ]:
con.sql("""SELECT date_trunc('year', date) yr, count(DISTINCT ticker) n FROM stocks GROUP BY 1 ORDER BY 1""").pl()